In [1]:
"""
Step 02: Live 1-Minute Bars During The Session

Objective:

    - Follow today's session minute by minute: each minute is requested STREAM_BAR_DELAY_SECONDS (30 s) after it ends,
      every poll covers the last STREAM_LOOKBACK_SECONDS (5 minutes) so a missed poll heals itself, and the first poll
      back-fills the session from the open.
    - The bars are kept in memory (stream.get_live_pdf()) and written to <staging>/live/ohlcv_live_YYYYMMDD.csv.

Prerequisites:

    - IB Gateway (or TWS) running and logged in, with a real-time data subscription for SPY.

Outputs:

    - <staging>/live/ohlcv_live_YYYYMMDD.csv. This file is NEVER merged into the raw folder: the record of a session is
      the after-close historical download (step01), which is checked minute by minute.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
# IMPORT THE INGEST CONFIGURATION
from ingest import config
# IMPORT THE IBKR CLIENT
from ingest.ibkr_client import IbkrApp
# IMPORT THE LIVE STREAM
from ingest.ibkr_stream import LiveBarStream, get_market_status_dict

In [3]:
# COLLECT THE MARKET STATUS (REAL NEW YORK TIME)
market_status_dict = get_market_status_dict()
# DISPLAY INFORMATION
pd.Series(market_status_dict)

now_ny_ts                  2026-10-06 14:05:18-04:00
session_today_bool                              True
market_open_bool                                True
first_bar_ts               2026-10-06 09:30:00-04:00
last_bar_ts                2026-10-06 15:59:00-04:00
next_first_bar_ts          2026-10-07 09:30:00-04:00
seconds_until_open_int                             0
seconds_until_close_int                         6882
dtype: object

In [4]:
# CONNECT WITH THE STREAM CLIENT ID (REUSE THE CONNECTION IF THIS CELL WAS ALREADY RUN)
try:
    # IF THE APPLICATION EXISTS AND IS CONNECTED
    connected_bool = stream_app.isConnected() or stream_app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_STREAM_CLIENT_ID_INT)
# IF THE APPLICATION DOES NOT EXIST YET
except NameError:
    # CREATE AND CONNECT IT
    stream_app = IbkrApp()
    connected_bool = stream_app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_STREAM_CLIENT_ID_INT)

⏳ Connecting to IBKR at 127.0.0.1:4001 (client id 2) ...
✅ Connected to IBKR (ibapi 10.45.1, server version 223)
ℹ️ IBKR [2104] Market data farm connection is OK:usfarm
ℹ️ IBKR [2106] HMDS data farm connection is OK:ushmds


ℹ️ IBKR [2158] Sec-def data farm connection is OK:secdefil
ℹ️ IBKR [2172] The version of the application you are running, 1045.1, needs to be upgraded, as it will be desupported on 20261215. The minimum supported version at that time will be 1050.1. 


In [5]:
# CREATE THE STREAM AND START IT (RUNS IN THE BACKGROUND; STOPS BY ITSELF AFTER THE LAST BAR)
stream = LiveBarStream(stream_app)
stream_started_bool = stream.start()

📡 Live stream for 2026-10-06: bars 09:30-15:59, each requested 30 s after it ends


In [6]:
# PREVIEW THE BARS RECEIVED SO FAR (RE-RUN THIS CELL TO REFRESH)
live_pdf = stream.get_live_pdf()
# DISPLAY INFORMATION
print(f"Running: {stream.is_running()} | bars: {len(live_pdf)} | last poll: {stream.poll_dict_list[-1] if stream.poll_dict_list else '-'}")
# PREVIEW DATAFRAME
live_pdf.tail(10)

Running: True | bars: 260 | last poll: {'poll_ts': Timestamp('2026-10-06 14:05:23-0400', tz='America/New_York'), 'window_str': '09:30-14:03', 'status_str': 'complete', 'bar_count_int': 260, 'total_bar_count_int': 260, 'missing_count_int': 14}


,timestamp,open,high,low,close,volume
250,2026-10-06 13:40:00-04:00,779.88,779.95,779.79,779.91,25924
251,2026-10-06 13:41:00-04:00,779.91,779.92,779.76,779.86,27115
252,2026-10-06 13:42:00-04:00,779.86,779.89,779.80,779.88,33916
253,2026-10-06 13:43:00-04:00,779.88,780.00,779.87,779.93,54270
254,2026-10-06 13:44:00-04:00,779.93,779.97,779.88,779.90,26052
255,2026-10-06 13:45:00-04:00,779.88,779.97,779.81,779.93,36037
256,2026-10-06 13:46:00-04:00,779.93,779.96,779.89,779.93,13081
257,2026-10-06 13:47:00-04:00,779.93,779.94,779.85,779.91,13792
258,2026-10-06 13:48:00-04:00,779.91,779.95,779.88,779.91,23119
259,2026-10-06 13:49:00-04:00,779.93,780.00,779.88,779.99,27602


In [7]:
# STOP THE STREAM
stream.stop()

⛔ Live stream stopped (260 bars)


In [8]:
# DISCONNECT
stream_app.disconnect_app()

✅ Disconnecting from IBKR ...
